## Load Gold Delta table

In [0]:
from pyspark.sql import functions as F


gold_zone_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "dimensions/taxi_zone"
)

df_zone = spark.read.format("delta").load(gold_zone_path)

print(f"Taxi zone dimension rows: {df_zone.count():,}")

display(
    df_zone
    .filter(F.col("is_current") == True)
    .orderBy("location_id")
)

### inspect the Silver schema once and confirm these columns exist exactly as expected

In [0]:
silver_path = (
    "abfss://silver@stnyctaxide2026nandi.dfs.core.windows.net/"
    "yellow_taxi/2026/*"
)

df_silver = spark.read.parquet(silver_path)

df_silver.printSchema()

## Test SCD2 zone joins

In [0]:
df_sample = df_silver.limit(10000)

pickup_zone = (
    df_zone
    .filter(F.col("is_current") == True)
    .select(
        F.col("zone_sk").alias("pickup_zone_sk"),
        F.col("location_id").alias("pickup_location_id")
    )
)

dropoff_zone = (
    df_zone
    .filter(F.col("is_current") == True)
    .select(
        F.col("zone_sk").alias("dropoff_zone_sk"),
        F.col("location_id").alias("dropoff_location_id")
    )
)

df_test = (
    df_sample
    .join(pickup_zone, "pickup_location_id", "left")
    .join(dropoff_zone, "dropoff_location_id", "left")
)

display(
    df_test.select(
        "pickup_location_id",
        "pickup_zone_sk",
        "dropoff_location_id",
        "dropoff_zone_sk",
        "pickup_date",
        "total_amount"
    ).limit(20)
)

In [0]:
print(
    "Unresolved pickup:",
    df_test.filter(F.col("pickup_zone_sk").isNull()).count()
)

print(
    "Unresolved dropoff:",
    df_test.filter(F.col("dropoff_zone_sk").isNull()).count()
)

In [0]:
df_fact_sample = (
    df_test
    .select(
        "pickup_date",
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "pickup_zone_sk",
        "dropoff_zone_sk",
        "passenger_count",
        "trip_distance",
        "trip_duration_minutes",
        "trip_speed_mph",
        "rate_code_id",
        "payment_type",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        "total_amount",
        "source_year",
        "source_month",
        "source_file",
        "ingestion_timestamp"
    )
)

print(f"Gold fact sample rows: {df_fact_sample.count():,}")

display(df_fact_sample.limit(20))

## validate the Gold fact sample

In [0]:
fact_validation = df_fact_sample.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("pickup_zone_sk").isNull(), 1).otherwise(0)).alias("null_pickup_zone_sk"),
    F.sum(F.when(F.col("dropoff_zone_sk").isNull(), 1).otherwise(0)).alias("null_dropoff_zone_sk"),
    F.sum(F.when(F.col("pickup_datetime").isNull(), 1).otherwise(0)).alias("null_pickup_datetime"),
    F.sum(F.when(F.col("total_amount").isNull(), 1).otherwise(0)).alias("null_total_amount"),
    F.sum(F.when(F.col("trip_distance") < 0, 1).otherwise(0)).alias("negative_distance"),
    F.sum(F.when(F.col("total_amount") < 0, 1).otherwise(0)).alias("negative_total_amount")
).collect()[0]

print(f"Rows: {fact_validation['rows']:,}")
print(f"Null pickup zone SK: {fact_validation['null_pickup_zone_sk']}")
print(f"Null dropoff zone SK: {fact_validation['null_dropoff_zone_sk']}")
print(f"Null pickup datetime: {fact_validation['null_pickup_datetime']}")
print(f"Null total amount: {fact_validation['null_total_amount']}")
print(f"Negative distance: {fact_validation['negative_distance']}")
print(f"Negative total amount: {fact_validation['negative_total_amount']}")

In [0]:
duplicate_count = (
    df_fact_sample
    .groupBy(
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "pickup_zone_sk",
        "dropoff_zone_sk",
        "total_amount"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate groups: {duplicate_count:,}")

## Create the final DataFrame and inspect its count/schema before writing

In [0]:
df_fact = (
    df_silver
    .join(
        pickup_zone,
        "pickup_location_id",
        "left"
    )
    .join(
        dropoff_zone,
        "dropoff_location_id",
        "left"
    )
    .select(
        "pickup_date",
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "pickup_zone_sk",
        "dropoff_zone_sk",
        "passenger_count",
        "trip_distance",
        "trip_duration_minutes",
        "trip_speed_mph",
        "rate_code_id",
        "payment_type",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        "total_amount",
        "source_year",
        "source_month",
        "source_file",
        "ingestion_timestamp"
    )
)

df_fact.printSchema()

In [0]:
print(f"Gold fact rows: {df_fact.count():,}")

## Gold fact write

In [0]:
gold_fact_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "facts/taxi_trip"
)

(
    df_fact
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("pickup_date")
    .save(gold_fact_path)
)

print("Gold fact written successfully.")

## validate Gold Delta

In [0]:
gold_fact_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "facts/taxi_trip"
)

df_gold_fact = (
    spark.read
    .format("delta")
    .load(gold_fact_path)
)

print(f"Gold fact rows: {df_gold_fact.count():,}")

In [0]:
df_gold_fact.select(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date")
).show()

In [0]:
df_gold_fact.filter(
    (F.col("pickup_date") < F.lit("2026-01-01")) |
    (F.col("pickup_date") > F.lit("2026-07-31"))
).select(
    "pickup_date",
    "source_year",
    "source_month",
    "source_file"
).limit(20).show(truncate=False)

## Skewed Data

In [0]:
gold_date_check = df_gold_fact.agg(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),

    F.sum(
        F.when(F.col("pickup_date") < F.lit("2026-01-01"), 1)
        .otherwise(0)
    ).alias("before_2026"),

    F.sum(
        F.when(F.col("pickup_date") > F.lit("2026-07-31"), 1)
        .otherwise(0)
    ).alias("after_july_2026"),

    F.sum(
        F.when(
            (F.col("source_month") == 7) &
            (F.col("pickup_date") > F.lit("2026-07-31")),
            1
        ).otherwise(0)
    ).alias("july_source_after_july"),

    F.sum(
        F.when(
            F.col("pickup_date") < F.lit("2026-01-01"),
            1
        ).otherwise(0)
    ).alias("pre_2026_rows")
).collect()[0]

print(f"Minimum pickup date       : {gold_date_check['min_pickup_date']}")
print(f"Maximum pickup date       : {gold_date_check['max_pickup_date']}")
print(f"Rows before 2026          : {gold_date_check['before_2026']:,}")
print(f"Rows after July 2026      : {gold_date_check['after_july_2026']:,}")
print(f"July source after July    : {gold_date_check['july_source_after_july']:,}")

### inspect the 53 records skew data

In [0]:
df_anomalies = (
    df_gold_fact
    .filter(
        (F.col("pickup_date") < F.lit("2026-01-01")) |
        (F.col("pickup_date") > F.lit("2026-07-31"))
    )
    .select(
        "pickup_date",
        "pickup_datetime",
        "dropoff_datetime",
        "vendor_id",
        "pickup_zone_sk",
        "dropoff_zone_sk",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "source_year",
        "source_month",
        "source_file"
    )
    .orderBy("pickup_date")
)

display(df_anomalies)

### quantify the 2001–2009 anomaly

In [0]:
df_gold_fact.filter(
    F.col("pickup_date") < F.lit("2025-12-01")
).groupBy(
    "pickup_date",
    "source_month",
    "source_file"
).count().orderBy("pickup_date").show(50, truncate=False)

In [0]:
df_gold_fact.filter(
    F.col("pickup_date") > F.lit("2026-07-31")
).groupBy(
    "pickup_date",
    "source_month",
    "source_file"
).count().orderBy("pickup_date").show(50, truncate=False)

## Create Gold anomaly quarantine

In [0]:
gold_anomaly_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "quarantine/taxi_trip_date_anomalies"
)

df_date_anomalies = (
    df_gold_fact
    .filter(F.col("pickup_date") < F.lit("2025-12-31"))
    .withColumn(
        "dq_reason",
        F.lit("Invalid historical pickup date")
    )
    .withColumn(
        "quarantine_timestamp",
        F.current_timestamp()
    )
)

print(f"Anomaly rows: {df_date_anomalies.count():,}")

In [0]:
# write Quarantine Table

(
    df_date_anomalies
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_anomaly_path)
)

print("Gold anomaly quarantine written successfully.")

## Rebuild Gold without those 10 rows

In [0]:
df_gold_fact_clean = (
    df_gold_fact
    .filter(F.col("pickup_date") >= F.lit("2025-12-31"))
)

In [0]:
(
    df_gold_fact_clean
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("pickup_date")
    .save(gold_fact_path)
)

print("Gold fact rebuilt successfully.")

In [0]:
spark.read.format("delta").load(gold_anomaly_path).count()

## Create the cleaned Gold DataFrame

In [0]:
df_gold_fact_clean = (
    df_gold_fact
    .filter(F.col("pickup_date") >= F.lit("2025-12-31"))
)

print(f"Clean Gold fact rows: {df_gold_fact_clean.count():,}")

## Overwrite Gold fact

In [0]:
(
    df_gold_fact_clean
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("pickup_date")
    .save(gold_fact_path)
)

print("Gold fact cleaned and written successfully.")

## Final lightweight `validation`

In [0]:
df_gold_fact_final = (
    spark.read
    .format("delta")
    .load(gold_fact_path)
)

df_gold_fact_final.agg(
    F.min("pickup_date").alias("min_pickup_date"),
    F.max("pickup_date").alias("max_pickup_date"),
    F.sum(
        F.when(F.col("pickup_date") < F.lit("2025-12-31"), 1)
        .otherwise(0)
    ).alias("invalid_historical_rows")
).show()

## Build first Gold analytical mart: daily_taxi_metrics

In [0]:
gold_fact_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "facts/taxi_trip"
)

df_gold_fact = (
    spark.read
    .format("delta")
    .load(gold_fact_path)
)

## Create the daily aggregation

In [0]:
from pyspark.sql import functions as F

df_daily_metrics = (
    df_gold_fact
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_trip_amount"),
        F.avg("fare_amount").alias("avg_fare_amount"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
        F.avg("tip_amount").alias("avg_tip_amount"),
        F.sum("passenger_count").alias("total_passengers")
    )
    .orderBy("pickup_date")
)

In [0]:
df_daily_metrics.printSchema()

display(df_daily_metrics.limit(20))

### clean the mart output

In [0]:
df_daily_metrics_final = (
    df_daily_metrics
    .select(
        "pickup_date",
        "total_trips",
        F.round("total_revenue", 2).alias("total_revenue"),
        F.round("avg_trip_amount", 2).alias("avg_trip_amount"),
        F.round("avg_fare_amount", 2).alias("avg_fare_amount"),
        F.round("avg_trip_distance", 2).alias("avg_trip_distance"),
        F.round("avg_trip_duration_minutes", 2).alias("avg_trip_duration_minutes"),
        F.round("avg_tip_amount", 2).alias("avg_tip_amount"),
        F.round("total_passengers", 0).cast("long").alias("total_passengers")
    )
)

display(df_daily_metrics_final.limit(20))

In [0]:
df_daily_metrics_final.agg(
    F.count("*").alias("days"),
    F.min("pickup_date").alias("min_date"),
    F.max("pickup_date").alias("max_date"),
    F.sum("total_trips").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue")
).show()

## Write the Gold daily mart

In [0]:
daily_mart_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/daily_taxi_metrics"
)

(
    df_daily_metrics_final
    .write
    .format("delta")
    .mode("overwrite")
    .save(daily_mart_path)
)

print("Daily taxi metrics mart written successfully.")

In [0]:
df_daily_mart = (
    spark.read
    .format("delta")
    .load(daily_mart_path)
)

df_daily_mart.agg(
    F.count("*").alias("days"),
    F.min("pickup_date").alias("min_date"),
    F.max("pickup_date").alias("max_date"),
    F.sum("total_trips").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue")
).show()

## Read persisted Gold fact and SCD2 dimension

In [0]:
from pyspark.sql import functions as F

gold_fact_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "facts/taxi_trip"
)

gold_zone_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "dimensions/taxi_zone"
)

df_gold_fact = (
    spark.read
    .format("delta")
    .load(gold_fact_path)
)

df_zone = (
    spark.read
    .format("delta")
    .load(gold_zone_path)
)

print(f"Gold fact rows: {df_gold_fact.count():,}")
print(f"Zone dimension rows: {df_zone.count():,}")

## Prepare the pickup-zone dimension

In [0]:
pickup_zone = (
    df_zone
    .filter(F.col("is_current") == True)
    .select(
        F.col("zone_sk").alias("pickup_zone_sk"),
        F.col("location_id").alias("pickup_location_id"),
        F.col("borough").alias("pickup_borough"),
        F.col("zone").alias("pickup_zone"),
        F.col("service_zone").alias("pickup_service_zone")
    )
)

## Build the zone performance aggregation

In [0]:
df_zone_performance = (
    df_gold_fact
    .join(
        pickup_zone,
        "pickup_zone_sk",
        "left"
    )
    .groupBy(
        "pickup_zone_sk",
        "pickup_location_id",
        "pickup_borough",
        "pickup_zone",
        "pickup_service_zone"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2)
         .alias("avg_trip_duration_minutes"),
        F.round(F.sum("passenger_count"), 0)
         .cast("long")
         .alias("total_passengers"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip_amount")
    )
)

In [0]:
df_zone_performance.printSchema()

display(
    df_zone_performance
    .orderBy(F.desc("total_trips"))
    .limit(20)
)

In [0]:
df_zone_performance.agg(
    F.count("*").alias("zone_count"),
    F.sum("total_trips").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue"),
    F.sum("total_passengers").alias("total_passengers")
).show()

## Write the Gold mart

In [0]:
zone_mart_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/zone_performance"
)

(
    df_zone_performance
    .write
    .format("delta")
    .mode("overwrite")
    .save(zone_mart_path)
)

print("Zone performance mart written successfully.")

In [0]:
df_zone_mart = (
    spark.read
    .format("delta")
    .load(zone_mart_path)
)

display(
    df_zone_mart.agg(
    F.count("*").alias("zone_count"),
    F.sum("total_trips").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue")
))

## Create the payment mart

In [0]:
df_payment_performance = (
    df_gold_fact
    .groupBy(
        "payment_type"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_trip_amount"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.sum("passenger_count"), 0)
         .cast("long")
         .alias("total_passengers")
    )
    .orderBy("payment_type")
)

display(df_payment_performance)

In [0]:
df_payment_performance.agg(
    F.count("*").alias("payment_type_count"),
    F.sum("total_trips").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue"),
    F.sum("total_passengers").alias("total_passengers")
).show()

##  Write Payment Performance Gold mart

In [0]:
payment_mart_path = (
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/payment_performance"
)

(
    df_payment_performance
    .write
    .format("delta")
    .mode("overwrite")
    .save(payment_mart_path)
)

print("Payment performance mart written successfully.")

In [0]:
df_payment_mart = (
    spark.read
    .format("delta")
    .load(payment_mart_path)
)

df_payment_mart.agg(
    F.count("*").alias("payment_type_count"),
    F.sum("total_trips").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue"),
    F.sum("total_passengers").alias("total_passengers")
).show()

## Gold final validation

In [0]:
# 1. Gold fact
df_fact_final = (
    spark.read
    .format("delta")
    .load(
        "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
        "facts/taxi_trip"
    )
)

print("Gold fact:", f"{df_fact_final.count():,}")

In [0]:
# 2. Taxi zone dimension
df_zone_final = (
    spark.read
    .format("delta")
    .load(
        "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
        "dimensions/taxi_zone"
    )
)

print(
    "Taxi zone dimension:",
    df_zone_final.count()
)

print(
    "Current dimension rows:",
    df_zone_final.filter(F.col("is_current") == True).count()
)

In [0]:
# 3. Gold marts
daily = spark.read.format("delta").load(
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/daily_taxi_metrics"
)

zone = spark.read.format("delta").load(
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/zone_performance"
)

payment = spark.read.format("delta").load(
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/payment_performance"
)

currency = spark.read.format("delta").load(
    "abfss://gold@stnyctaxide2026nandi.dfs.core.windows.net/"
    "marts/daily_taxi_metrics_usd_inr"
)

print("Daily mart:", daily.count())
print("Zone mart:", zone.count())
print("Payment mart:", payment.count())
print("Currency mart:", currency.count())